# Notebook 04 — Model Evaluation
SkyGuard AI | Conflict Detection + Trajectory Prediction Assessment

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../backend"))
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from src.conflict_detector import haversine, detect_conflicts
from src.forecasting import predict_linear_path
print("Imports OK")

## 1. Verify Haversine Implementation
Ground truth: Delhi to Mumbai = ~1147 km

In [ ]:
dist = haversine(28.6139, 77.2090, 19.0760, 72.8777)
print(f"Delhi to Mumbai: {dist:.1f} km (expected ~1147 km)")
assert 1100 < dist < 1200, "Haversine result out of expected range!"
print("Haversine verified")

## 2. Conflict Detection Benchmark

In [ ]:
df = pd.read_csv("../backend/data/cleaned_data.csv")
conflicts = detect_conflicts(df, threshold_km=10)
print(f"Total flights: {len(df)}")
print(f"Conflict pairs detected: {len(conflicts)}")
if conflicts:
    row, other, dist = conflicts[0]
    print(f"Example: {row['icao24']} vs {other['icao24']} — {dist:.2f} km")

## 3. Threshold Sensitivity Analysis

In [ ]:
results = []
for threshold in [5, 10, 15, 20, 25, 30]:
    n = len(detect_conflicts(df, threshold_km=threshold))
    results.append({"threshold_km": threshold, "conflicts": n})
results_df = pd.DataFrame(results)
fig = px.line(results_df, x="threshold_km", y="conflicts",
    title="Conflict Count vs Distance Threshold",
    markers=True)
fig.show()

## 4. Trajectory Prediction Evaluation

In [ ]:
history = [
    {"lat": 28.61, "lng": 77.21},
    {"lat": 28.60, "lng": 77.35},
    {"lat": 28.59, "lng": 77.49},
    {"lat": 28.58, "lng": 77.63},
    {"lat": 28.57, "lng": 77.77},
]
predictions = predict_linear_path(
    history=history,
    current_lat=28.57, current_lng=77.77,
    heading=90, velocity=250, steps=5
)
for p in predictions:
    print(f"Step {p['step']}: lat={p['lat']:.4f}, lng={p['lng']:.4f}")

## 5. Visualize Predicted Path

In [ ]:
all_pts = [{"lat": h["lat"], "lng": h["lng"], "type": "history"} for h in history]
for p in predictions:
    all_pts.append({"lat": p["lat"], "lng": p["lng"], "type": "predicted"})
pdf = pd.DataFrame(all_pts)
fig = px.scatter_geo(pdf, lat="lat", lon="lng", color="type",
    title="Trajectory: History vs Predicted Path",
    projection="natural earth")
fig.show()

## 6. Summary

| Module | Status | Notes |
|---|---|---|
| Haversine | Verified | Delhi-Mumbai test passed |
| Conflict Detection | Running | Threshold-sensitive |
| Heuristic Trajectory | Running | Bearing extrapolation |
| LSTM Trajectory | Pending | Run Notebook 03 first |